# 01 — Introduction (HTML)

A guided tour of the HTML builder. Each section is a small
self-contained example showing the code, its rendered source,
and the visual result.

**Topics:**
1. Hello World — minimal page with `HtmlBuilderHandler`.
2. Lifecycle — `create()` / `build()` / `render()` and the
   distinction between `source` and `built`.
3. Pretty-print — readable indented HTML with `pretty=True`.
4. Void tags & `xml=False` — XHTML self-close vs idiomatic HTML5.
5. Three-state booleans — `True`/`False` as JS literals.
6. Keyword-collision attributes — `_class` → `class`.

## 1. Hello World

A page is a subclass of `HtmlBuilderHandler` that implements
`main(self, root)`. `root` is the `source` Bag where the page
recipe is written.

The lifecycle is explicit: instantiate the handler, then call
`create()`, `build()`, `render()` in order.

In [ ]:
from IPython.display import HTML

from genro_builders.contrib.html import HtmlBuilderHandler


class HelloPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.h1("Hello World")
        body.p("My first page with genro-builders.")


page = HelloPage()
page.create()
page.build()
print(page.render())

In [ ]:
HTML(page.render())

## 2. Lifecycle: source vs built

`create()` calls `main(self.source)` and populates the `source`
Bag — the literal recipe the user wrote. `build()` materialises
the source into the `built` Bag. For elementary pages the built
is a 1:1 mirror of the source; the distinction matters when
components, iterate, or pointers come into play (decision 7 of
the architecture contract — components stay opaque in source).

Both bags can be inspected with `to_xml()`.

In [ ]:
print("--- source ---")
print(page.source.to_xml())
print("--- built ---")
print(page.built.to_xml())

## 3. Pretty-print

By default `render()` returns a single linear string — compact
and ideal for production output. For readable output during
development pass `pretty=True`: the renderer indents with 2
spaces, puts each element on its own line, and keeps text-only
leaves on a single line.

In [ ]:
print(page.render(pretty=True))

In [ ]:
HTML(page.render(pretty=True))

## 4. Void tags and `xml=False`

HTML5 void tags (`img`, `br`, `hr`, `input`, ...) cannot have
children. By default they are emitted self-close XHTML-style
(`<img src="x"/>`) so the document is also XML well-formed —
useful for pipelines that mix HTML and SVG. With `xml=False`
the output is idiomatic HTML5 (`<img src="x">`).

In [ ]:
class LogoPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.img(src="logo.png", alt="Logo")
        body.br()
        body.p("Below the logo.")


logo = LogoPage()
logo.create()
logo.build()

print("xml=True  (default):", logo.render())
print("xml=False (HTML5):  ", logo.render(xml=False))

In [ ]:
HTML(logo.render(pretty=True))

## 5. Three-state boolean attributes

Boolean attributes are serialised as JS literals so the
client-side code can consume them directly: `True` → `"true"`,
`False` → `"false"`. The value `None` is currently filtered
upstream (see `tests/test_html_render.py`); a per-node opt-in
to emit `"null"` will be added later.

In [ ]:
class FormPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.input(type="text", disabled=True)
        body.input(type="checkbox", checked=False)


form = FormPage()
form.create()
form.build()
print(form.render(pretty=True))

In [ ]:
HTML(form.render())

## 6. Keyword-collision attributes

Python keywords (`class`, `for`) cannot be used as argument
names, so the convention is to prefix them with an underscore:
`_class`, `_for`. The renderer maps them back to their HTML
names automatically.

In [ ]:
class StyledPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.div("Important notice", _class="alert primary")


styled = StyledPage()
styled.create()
styled.build()
print(styled.render(pretty=True))

In [ ]:
HTML(styled.render())